# Radiomics, Deep Learning & Feature Fusion for Breast Lesion Classification (CBIS-DDSM-R)

**Question.** Does fusing hand-crafted *radiomics* features with *deep* CNN / ViT features improve benign-vs-malignant
classification of mammographic lesions, compared with either feature family alone?

**Data.**
- **Radiomics + metadata:** [CBIS-DDSM-R](https://huggingface.co/datasets/helloerikaaa/cbis-ddsm-r) — 2,437 lesion ROIs with
  93 PyRadiomics features (first-order, GLCM, GLDM, GLRLM, GLSZM, NGTDM).
- **Images:** cropped lesion ROIs from the Kaggle JPEG release of CBIS-DDSM
  ([awsaf49/cbis-ddsm-breast-cancer-image-dataset](https://www.kaggle.com/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset)),
  matched to the radiomics rows through the ROI series UID (100 % match).
- **Label:** `MALIGNANT` = 1; `BENIGN` and `BENIGN_WITHOUT_CALLBACK` = 0.

**Pipeline.**
1. Patient-level train / held-out test split (no patient appears in both).
2. Radiomics baseline (XGBoost, SVM, logistic regression) and feature selection (LASSO, RFE, consensus) — selection runs *inside* each CV fold.
3. Frozen deep features from 5 ImageNet backbones (ResNet50, DenseNet121, EfficientNetV2-S, Swin-T, DeiT-S) on CLAHE-enhanced ROIs.
4. Early fusion (radiomics ⊕ deep features → XGBoost) and late fusion (averaged probabilities).
5. Partial fine-tuning of the best backbone on the training patients only.
6. **One** final evaluation on the untouched test set: AUC with patient-level bootstrap 95 % CIs, paired bootstrap tests for the fusion gain, subgroup analysis and feature-importance breakdown.

**Evaluation protocol.** All model and feature choices are made with 5-fold *patient-grouped* cross-validation on the training set.
The test set is used exactly once, in Section 7.

## 1. Setup

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
SEED        = 42
HF_DATASET  = "helloerikaaa/cbis-ddsm-r"        # radiomics + clinical metadata
LOCAL_CSV   = None                              # optional: path to a local copy of the CBIS-DDSM-R csv
IMG_ROOT    = "/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset"
OUT_DIR     = "outputs"                         # figures, cached features, result tables
PRETRAINED  = True                              # ImageNet weights for every backbone

BACKBONES = {                                   # display name → timm model id
    "ResNet50":         "resnet50",
    "DenseNet121":      "densenet121",
    "EfficientNetV2-S": "tf_efficientnetv2_s",
    "Swin-T":           "swin_tiny_patch4_window7_224",
    "DeiT-S":           "deit_small_patch16_224",
}

IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2
FT_EPOCHS   = 15
FT_LR       = 1e-4
N_BOOT      = 1000                              # bootstrap resamples for confidence intervals

In [ ]:
try:
    import timm
except ImportError:
    %pip install -q timm
    import timm

In [ ]:
import os, glob, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
from PIL import Image
from IPython.display import display, Markdown
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

from sklearn.base import BaseEstimator, clone
from sklearn.feature_selection import SelectorMixin, SelectFromModel, RFE
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV, LogisticRegression
from sklearn.metrics import (roc_auc_score, roc_curve, f1_score, accuracy_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*mismatched devices.*")   # xgboost GPU model + numpy input

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
XGB_DEVICE = "cuda" if DEVICE == "cuda" else "cpu"
os.makedirs(f"{OUT_DIR}/features", exist_ok=True)

# Chart colours (one hue per feature family, fixed order)
C_RAD, C_DEEP, C_FUSE, C_MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 110})

def savefig(name):
    plt.savefig(f"{OUT_DIR}/{name}", dpi=150, bbox_inches="tight")

print(f"Device: {DEVICE} | torch {torch.__version__} | timm {timm.__version__}")

In [ ]:
# The dataset is public; a Hugging Face token only avoids anonymous rate limits.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HUGGING_FACE_TOKEN")
except Exception:
    pass

## 2. Data

### 2.1 Load radiomics table and define the label

In [ ]:
if LOCAL_CSV:
    df = pd.read_csv(LOCAL_CSV)
else:
    from datasets import load_dataset
    df = load_dataset(HF_DATASET, split="train").to_pandas()

# cropped_image_file_path = CaseName/StudyUID/SeriesUID/000001.dcm (with a trailing newline)
df["cropped_image_file_path"] = df["cropped_image_file_path"].astype(str).str.strip()
path_parts    = df["cropped_image_file_path"].str.split("/")
df["case_id"] = path_parts.str[0]            # e.g. Calc-Training_P_00005_RIGHT_MLO_1
df["roi_uid"] = path_parts.str[2]            # series UID of the cropped ROI → joins to the image files
df["label"]   = (df["pathology"] == "MALIGNANT").astype(int)

RADIOMICS_PREFIXES = ("original_firstorder_", "original_glcm_", "original_gldm_",
                      "original_glrlm_", "original_glszm_", "original_ngtdm_")
radiomics_cols = [c for c in df.columns if c.startswith(RADIOMICS_PREFIXES)]

print(f"Loaded {len(df):,} ROIs × {df.shape[1]} columns | {len(radiomics_cols)} radiomics features")
print(df["pathology"].value_counts().to_string())

### 2.2 Match each ROI to its cropped image

In [ ]:
def find_img_root(default):
    if os.path.exists(f"{default}/csv/dicom_info.csv"):
        return default
    hits = glob.glob("/kaggle/input/**/csv/dicom_info.csv", recursive=True)
    if not hits:
        raise FileNotFoundError("Attach the Kaggle dataset 'awsaf49/cbis-ddsm-breast-cancer-image-dataset'.")
    return os.path.dirname(os.path.dirname(hits[0]))

IMG_ROOT   = find_img_root(IMG_ROOT)
dicom_info = pd.read_csv(f"{IMG_ROOT}/csv/dicom_info.csv")

# image_path = CBIS-DDSM/jpeg/<SeriesUID>/1-xxx.jpg
cropped = dicom_info[dicom_info["SeriesDescription"] == "cropped images"].copy()
cropped["roi_uid"]  = cropped["image_path"].str.split("/").str[2]
cropped["img_path"] = cropped["image_path"].str.replace("CBIS-DDSM/jpeg", f"{IMG_ROOT}/jpeg", regex=False)
image_map = cropped.drop_duplicates("roi_uid").set_index("roi_uid")["img_path"]

df["img_path"] = df["roi_uid"].map(image_map)
n_missing = int(df["img_path"].isna().sum())
print(f"Matched {len(df) - n_missing:,} / {len(df):,} ROIs to a cropped JPEG")
if n_missing:
    print(f"Dropping {n_missing} unmatched ROIs")
    df = df[df["img_path"].notna()].reset_index(drop=True)

assert df["img_path"].map(os.path.exists).all(), "some matched image files are missing on disk"

### 2.3 Cohort overview

In [ ]:
cohort = pd.crosstab(df["abnormality_type"], df["label"].map({0: "Benign", 1: "Malignant"}), margins=True)
display(cohort)

per_patient = df.groupby("patient_id").size()
print(f"{df['patient_id'].nunique():,} patients | ROIs per patient: median {per_patient.median():.0f}, max {per_patient.max()}")
print("Several ROIs per patient (CC + MLO views of the same lesion, multiple lesions) → splits must be grouped by patient.")

### 2.4 Patient-level train / test split

CBIS-DDSM contains the CC **and** MLO view of the same lesion, so a row-level random split puts near-duplicate images
of one lesion on both sides and inflates every metric. All splits below are grouped by `patient_id`.

In [ ]:
y_all      = df["label"].to_numpy()
groups_all = df["patient_id"].to_numpy()

outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)   # ≈ 80 / 20
train_idx, test_idx = next(outer.split(df, y_all, groups_all))
assert set(groups_all[train_idx]).isdisjoint(groups_all[test_idx])

y_tr, y_te = y_all[train_idx], y_all[test_idx]
g_tr, g_te = groups_all[train_idx], groups_all[test_idx]

# Cross-validation used for every model-selection decision (training patients only)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

SPW = (y_tr == 0).sum() / (y_tr == 1).sum()     # XGBoost scale_pos_weight, from the training set only

display(pd.DataFrame({
    "ROIs":        [len(train_idx), len(test_idx)],
    "Patients":    [len(set(g_tr)), len(set(g_te))],
    "Malignant %": [100 * y_tr.mean(), 100 * y_te.mean()],
}, index=["Train", "Test"]).round(1))
print(f"scale_pos_weight = {SPW:.3f}")

### 2.5 Shared modelling helpers

In [ ]:
def make_xgb():
    return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         scale_pos_weight=SPW, eval_metric="logloss",
                         tree_method="hist", device=XGB_DEVICE,
                         random_state=SEED, n_jobs=-1)

def xgb_pipeline(selector=None):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if selector is not None:
        steps += [("scale", StandardScaler()), ("select", selector)]
    steps.append(("clf", make_xgb()))
    return Pipeline(steps)

def linear_pipeline(clf):
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()), ("clf", clf)])

METRICS = {"AUC": "roc_auc", "F1": "f1", "Accuracy": "accuracy"}

def cv_evaluate(name, model, X, **info):
    # Patient-grouped 5-fold CV on the training set. Every preprocessing step is refit inside each fold.
    res = cross_validate(model, X, y_tr, groups=g_tr, cv=cv, scoring=list(METRICS.values()), n_jobs=1)
    row = {"Model": name, **info}
    for label, key in METRICS.items():
        row[label], row[f"{label}_sd"] = res[f"test_{key}"].mean(), res[f"test_{key}"].std()
    return row

def show_cv(rows):
    t = pd.DataFrame(rows).sort_values("AUC", ascending=False).reset_index(drop=True)
    shown = t.drop(columns=[c for c in t if c.endswith("_sd")]).copy()
    for m in METRICS:
        shown[m] = [f"{a:.3f} ± {s:.3f}" for a, s in zip(t[m], t[f"{m}_sd"])]
    display(shown)
    return t

## 3. Radiomics

### 3.1 Baseline classifiers (all 93 features)

In [ ]:
X_rad_all = df[radiomics_cols].to_numpy(dtype=float)
X_rad_tr  = X_rad_all[train_idx]

baseline_cv = show_cv([
    cv_evaluate("XGBoost", xgb_pipeline(), X_rad_tr),
    cv_evaluate("SVM (RBF)", linear_pipeline(SVC(kernel="rbf", class_weight="balanced", random_state=SEED)), X_rad_tr),
    cv_evaluate("Logistic regression", linear_pipeline(LogisticRegression(class_weight="balanced", max_iter=2000)), X_rad_tr),
])

### 3.2 Feature selection

Three strategies are compared: **LASSO** (L1-penalised regression, α chosen by internal CV), **RFE** with logistic
regression (top 20), and their **consensus** (features chosen by both). The selector is a step of the pipeline,
so it is refit inside every CV fold and never sees the validation fold.

In [ ]:
class ConsensusSelector(SelectorMixin, BaseEstimator):
    # Keeps only the features chosen by every selector in `selectors`.
    def __init__(self, selectors=()):
        self.selectors = selectors

    def fit(self, X, y=None):
        self.fitted_  = [clone(s).fit(X, y) for s in self.selectors]
        self.support_ = np.logical_and.reduce([s.get_support() for s in self.fitted_])
        if not self.support_.any():
            raise ValueError("selectors share no features")
        return self

    def _get_support_mask(self):
        return self.support_

def lasso_selector():
    return SelectFromModel(LassoCV(cv=5, max_iter=10000, random_state=SEED))

def rfe_selector(k=20):
    return RFE(LogisticRegression(class_weight="balanced", max_iter=2000), n_features_to_select=k, step=5)

SELECTORS = {
    "LASSO":                    lasso_selector,
    "RFE-LR (k=20)":            rfe_selector,
    "Consensus (LASSO ∩ RFE)":  lambda: ConsensusSelector([lasso_selector(), rfe_selector()]),
}

# Fit each selector once on the full training set → the feature list it would hand on
selected = {}
for name, make in SELECTORS.items():
    p = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()),
                  ("select", make())]).fit(X_rad_tr, y_tr)
    selected[name] = [c for c, keep in zip(radiomics_cols, p["select"].get_support()) if keep]
selected["All features"] = radiomics_cols

fs_rows = [cv_evaluate("All features", xgb_pipeline(), X_rad_tr, n_features=len(radiomics_cols))]
for name, make in SELECTORS.items():
    fs_rows.append(cv_evaluate(name, xgb_pipeline(make()), X_rad_tr, n_features=len(selected[name])))
fs_cv = show_cv(fs_rows)

print("Consensus features:", selected["Consensus (LASSO ∩ RFE)"])

In [ ]:
best_fs   = fs_cv.loc[0, "Model"]
best_cols = selected[best_fs]
X_rad_best = df[best_cols].to_numpy(dtype=float)       # used by every radiomics / fusion arm below

fig, ax = plt.subplots(figsize=(8, 3.2))
order = fs_cv.iloc[::-1]
colors = [C_RAD if m == best_fs else C_MUTED for m in order["Model"]]
ax.barh([f"{m}  (n={n})" for m, n in zip(order["Model"], order["n_features"])], order["AUC"],
        xerr=order["AUC_sd"], color=colors, height=0.55, capsize=3, error_kw={"elinewidth": 1})
for i, v in enumerate(order["AUC"]):
    ax.text(v + order["AUC_sd"].iloc[i] + 0.004, i, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlim(0.5, max(order["AUC"]) + 0.06)
ax.set_xlabel("AUC (mean ± SD, 5-fold patient-grouped CV on training set)")
ax.set_title("Radiomics feature selection — XGBoost", loc="left")
ax.grid(axis="y", visible=False)
plt.tight_layout(); savefig("fig_feature_selection.png"); plt.show()

print(f"Selected: {best_fs} → {len(best_cols)} radiomics features carried forward")

## 4. Deep features

### 4.1 Image pipeline

ROIs are contrast-enhanced with **CLAHE** (clip 2.0, 8×8 tiles), replicated to 3 channels, resized to 224×224 and
ImageNet-normalised. Unreadable images raise an error instead of silently becoming black images.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def load_roi(path, clahe=True):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    if clahe:
        img = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)
    return Image.fromarray(img).convert("RGB")

class ROIDataset(Dataset):
    def __init__(self, frame, transform, clahe=True):
        self.paths, self.labels = frame["img_path"].tolist(), frame["label"].tolist()
        self.transform, self.clahe = transform, clahe

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(load_roi(self.paths[i], self.clahe)), self.labels[i]

def make_loader(frame, transform, shuffle=False, batch_size=BATCH_SIZE):
    return DataLoader(ROIDataset(frame, transform), batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"),
                      generator=torch.Generator().manual_seed(SEED) if shuffle else None)

In [ ]:
samples = df.groupby("label").sample(3, random_state=SEED).sort_values("label")
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for col, (_, row) in enumerate(samples.iterrows()):
    tag = "Malignant" if row["label"] else "Benign"
    for r, clahe in enumerate([False, True]):
        ax = axes[r, col]
        ax.imshow(load_roi(row["img_path"], clahe=clahe))
        ax.set_title(f"{tag} · {row['abnormality_type']}\n{'CLAHE' if clahe else 'original'}", fontsize=8)
        ax.axis("off")
plt.suptitle("Sample ROIs — original (top) vs CLAHE (bottom)", fontsize=12)
plt.tight_layout(); savefig("fig_sample_rois.png"); plt.show()

### 4.2 Frozen backbone features

Each backbone is used as a fixed feature extractor (`num_classes=0` → pooled embedding). Transformers use their
default pooling so the pretrained final norm is kept. No labels are involved, so extracting features for all ROIs
at once does not leak test information. Features are cached in `outputs/features/`.

In [ ]:
def create_backbone(timm_name, num_classes=0):
    return timm.create_model(timm_name, pretrained=PRETRAINED, num_classes=num_classes)

@torch.no_grad()
def extract_features(model, frame):
    model.eval().to(DEVICE)
    feats = [model(x.to(DEVICE)).float().cpu().numpy()
             for x, _ in tqdm(make_loader(frame, eval_tf), leave=False)]
    return np.concatenate(feats)

def cached_features(tag, compute):
    path = f"{OUT_DIR}/features/{tag}.npz"
    if os.path.exists(path):
        z = np.load(path, allow_pickle=True)
        if np.array_equal(z["roi_uid"], df["roi_uid"].to_numpy()):
            return z["feats"]
    feats = compute()
    np.savez(path, feats=feats, roi_uid=df["roi_uid"].to_numpy())
    return feats

def free(model):
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

deep_feats = {}
for name, timm_name in BACKBONES.items():
    def compute():
        model = create_backbone(timm_name)
        feats = extract_features(model, df)
        free(model)
        return feats
    deep_feats[name] = cached_features(f"{timm_name}_frozen", compute)
    print(f"{name:<17} {deep_feats[name].shape}")

## 5. Frozen features: deep vs early fusion (training-set CV)

**Early fusion** = concatenating the selected radiomics features with the deep embedding and training one XGBoost model.

In [ ]:
X_rad_best_tr = X_rad_best[train_idx]
fusion_rows = [cv_evaluate("Radiomics", xgb_pipeline(), X_rad_best_tr,
                           Family="Radiomics", Backbone="—", Dim=X_rad_best.shape[1])]
for name, F in deep_feats.items():
    fusion_rows.append(cv_evaluate(name, xgb_pipeline(), F[train_idx],
                                   Family="Deep", Backbone=name, Dim=F.shape[1]))
    fused = np.hstack([X_rad_best, F])
    fusion_rows.append(cv_evaluate(f"{name} + Radiomics", xgb_pipeline(), fused[train_idx],
                                   Family="Fusion", Backbone=name, Dim=fused.shape[1]))
fusion_cv = show_cv(fusion_rows)

In [ ]:
names = list(BACKBONES)
deep_auc = fusion_cv[fusion_cv.Family == "Deep"].set_index("Backbone").loc[names]
fuse_auc = fusion_cv[fusion_cv.Family == "Fusion"].set_index("Backbone").loc[names]
rad_auc  = fusion_cv.loc[fusion_cv.Family == "Radiomics", "AUC"].item()

x, w = np.arange(len(names)), 0.36
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w/2 - 0.01, deep_auc["AUC"], w, yerr=deep_auc["AUC_sd"], color=C_DEEP, label="Deep only", capsize=3)
ax.bar(x + w/2 + 0.01, fuse_auc["AUC"], w, yerr=fuse_auc["AUC_sd"], color=C_FUSE, label="Deep + radiomics", capsize=3)
ax.axhline(rad_auc, color=C_RAD, lw=2, ls="--", label=f"Radiomics only ({rad_auc:.3f})")
ax.set_xticks(x, names)
ax.set_ylim(0.5, min(1.0, max((deep_auc.AUC + deep_auc.AUC_sd).max(), (fuse_auc.AUC + fuse_auc.AUC_sd).max(), rad_auc) + 0.04))
ax.set_ylabel("AUC (5-fold CV, training set)")
fig.suptitle("Frozen backbones: does adding radiomics help?", x=0.01, ha="left")
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=3); ax.grid(axis="x", visible=False)
plt.tight_layout(); savefig("fig_frozen_fusion_cv.png"); plt.show()

best_backbone = deep_auc["AUC"].idxmax()
print(f"Best frozen backbone (deep-only CV AUC): {best_backbone} → fine-tuned in Section 6")

## 6. Fine-tuning the best backbone

The last stage + classifier head are unfrozen and trained on the **training patients only** (with an inner,
patient-grouped validation split for checkpoint selection). Frozen BatchNorm layers are kept in eval mode so their
running statistics stay at their ImageNet values. Test patients are never seen.

In [ ]:
def unfreeze_last_stage(model, timm_name):
    for p in model.parameters():
        p.requires_grad = False
    if "resnet" in timm_name:
        modules = [model.layer4]
    elif "densenet" in timm_name:
        modules = [model.features.denseblock4, model.features.norm5]
    elif "efficientnet" in timm_name:
        modules = [model.blocks[-1], model.conv_head, model.bn2]   # last stage
    elif "swin" in timm_name:
        modules = [model.layers[-1], model.norm]
    else:                                              # ViT / DeiT
        modules = [*list(model.blocks)[-2:], model.norm]
    modules.append(model.get_classifier())
    for m in modules:
        for p in m.parameters():
            p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.1f}%)")

def freeze_frozen_bn(model):
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in m.parameters()):
            m.eval()

ft_timm = BACKBONES[best_backbone]
seed_everything(SEED)
ft_model = create_backbone(ft_timm, num_classes=2).to(DEVICE)
print(f"Fine-tuning {best_backbone} ({ft_timm})")
unfreeze_last_stage(ft_model, ft_timm)

# Inner patient-grouped split of the TRAINING set: ~85 % fit / ~15 % validation
inner = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
fit_rel, val_rel = next(inner.split(train_idx, y_tr, g_tr))
fit_df, val_df   = df.iloc[train_idx[fit_rel]], df.iloc[train_idx[val_rel]]
print(f"Fit: {len(fit_df)} ROIs | Validation: {len(val_df)} ROIs")

In [ ]:
@torch.no_grad()
def predict_proba_cnn(model, frame):
    model.eval()
    probs = [torch.softmax(model(x.to(DEVICE)), 1)[:, 1].float().cpu().numpy()
             for x, _ in make_loader(frame, eval_tf)]
    return np.concatenate(probs)

fit_loader = make_loader(fit_df, train_tf, shuffle=True, batch_size=16)
criterion  = nn.CrossEntropyLoss(weight=torch.tensor([1.0, SPW], dtype=torch.float32, device=DEVICE))
optimizer  = torch.optim.AdamW([p for p in ft_model.parameters() if p.requires_grad], lr=FT_LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FT_EPOCHS)

history, best_val_auc, best_state = {"train_loss": [], "val_auc": []}, -np.inf, None
for epoch in range(FT_EPOCHS):
    ft_model.train(); freeze_frozen_bn(ft_model)
    running = 0.0
    for imgs, labels in fit_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(ft_model(imgs), labels)
        loss.backward()
        optimizer.step()
        running += loss.item()
    scheduler.step()

    val_auc = roc_auc_score(val_df["label"], predict_proba_cnn(ft_model, val_df))
    history["train_loss"].append(running / len(fit_loader))
    history["val_auc"].append(val_auc)
    flag = ""
    if val_auc > best_val_auc:
        best_val_auc, flag = val_auc, "  ← best"
        best_state = {k: v.detach().cpu().clone() for k, v in ft_model.state_dict().items()}
    print(f"Epoch {epoch + 1:02d}/{FT_EPOCHS}  loss {history['train_loss'][-1]:.4f}  val AUC {val_auc:.4f}{flag}")

ft_model.load_state_dict(best_state)
torch.save(best_state, f"{OUT_DIR}/finetuned_{ft_timm}.pt")
print(f"Best validation AUC: {best_val_auc:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
epochs = np.arange(1, len(history["train_loss"]) + 1)
ax1.plot(epochs, history["train_loss"], marker="o", ms=4, lw=2, color=C_DEEP)
ax1.set(title="Training loss", xlabel="Epoch", ylabel="Weighted cross-entropy")
ax2.plot(epochs, history["val_auc"], marker="o", ms=4, lw=2, color=C_FUSE)
best_ep = int(np.argmax(history["val_auc"])) + 1
ax2.scatter([best_ep], [max(history["val_auc"])], s=90, facecolors="none", edgecolors="black", zorder=3)
ax2.set(title="Validation AUC (inner split)", xlabel="Epoch", ylabel="AUC")
plt.suptitle(f"Fine-tuning {best_backbone}", x=0.01, ha="left")
plt.tight_layout(); savefig("fig_finetune_history.png"); plt.show()

In [ ]:
# End-to-end CNN predictions on the test set, then embeddings from the fine-tuned network
p_e2e_test = predict_proba_cnn(ft_model, df.iloc[test_idx])
ft_model.reset_classifier(0)
ft_feats = extract_features(ft_model, df)
free(ft_model)
print(f"Fine-tuned embedding: {ft_feats.shape}")

## 7. Final evaluation on the held-out test set

Every arm is refit on the full training set and scored once on the test patients.
AUC 95 % CIs come from a **patient-level** bootstrap (resampling patients, not ROIs).
Sensitivity / specificity / F1 / accuracy use a 0.5 threshold.

| Arm | Features | Classifier |
|---|---|---|
| Radiomics | selected radiomics | XGBoost |
| *X* (frozen) | frozen backbone embedding | XGBoost |
| *X* + Radiomics | early fusion (concatenation) | XGBoost |
| *best* fine-tuned (end-to-end) | — | the fine-tuned CNN's own softmax |
| *best* fine-tuned (features) | fine-tuned embedding | XGBoost |
| *best* fine-tuned + Radiomics | early fusion | XGBoost |
| Late fusion | mean of two arms' probabilities | — |

In [ ]:
test_probs, arm_family, fitted = {}, {}, {}

def add_arm(name, family, X=None, proba=None):
    if proba is None:
        model = xgb_pipeline().fit(X[train_idx], y_tr)
        fitted[name] = model
        proba = model.predict_proba(X[test_idx])[:, 1]
    test_probs[name], arm_family[name] = proba, family

add_arm("Radiomics", "Radiomics", X_rad_best)
for name, F in deep_feats.items():
    add_arm(f"{name} (frozen)", "Deep", F)
    add_arm(f"{name} (frozen) + Radiomics", "Early fusion", np.hstack([X_rad_best, F]))

FT = f"{best_backbone} fine-tuned"
add_arm(f"{FT} (end-to-end)", "Deep", proba=p_e2e_test)
add_arm(f"{FT} (features)", "Deep", ft_feats)
add_arm(f"{FT} + Radiomics", "Early fusion", np.hstack([X_rad_best, ft_feats]))
add_arm(f"Late fusion: {best_backbone} (frozen) + Radiomics", "Late fusion",
        proba=(test_probs[f"{best_backbone} (frozen)"] + test_probs["Radiomics"]) / 2)
add_arm(f"Late fusion: {FT} + Radiomics", "Late fusion",
        proba=(test_probs[f"{FT} (end-to-end)"] + test_probs["Radiomics"]) / 2)
print(f"{len(test_probs)} arms evaluated on {len(test_idx)} test ROIs")

In [ ]:
# Patient-level bootstrap resamples, shared by all arms so comparisons are paired
rng = np.random.default_rng(SEED)
patients = np.unique(g_te)
rows_of  = {p: np.flatnonzero(g_te == p) for p in patients}
BOOT = []
while len(BOOT) < N_BOOT:
    b = np.concatenate([rows_of[p] for p in rng.choice(patients, len(patients), replace=True)])
    if 0 < y_te[b].mean() < 1:
        BOOT.append(b)

def boot_aucs(p):
    return np.array([roc_auc_score(y_te[b], p[b]) for b in BOOT])

def summarize(name, p, thr=0.5):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    lo, hi = np.percentile(boot_aucs(p), [2.5, 97.5])
    return {"Model": name, "Family": arm_family[name], "AUC": roc_auc_score(y_te, p),
            "AUC_lo": lo, "AUC_hi": hi, "Sensitivity": tp / (tp + fn), "Specificity": tn / (tn + fp),
            "F1": f1_score(y_te, pred), "Accuracy": accuracy_score(y_te, pred)}

results = (pd.DataFrame([summarize(n, p) for n, p in test_probs.items()])
             .sort_values("AUC", ascending=False).reset_index(drop=True))
results.to_csv(f"{OUT_DIR}/results_test.csv", index=False)

table = results.assign(**{"AUC (95% CI)": [f"{a:.3f} ({l:.3f}–{h:.3f})"
                                           for a, l, h in zip(results.AUC, results.AUC_lo, results.AUC_hi)]})
table = table[["Model", "Family", "AUC (95% CI)", "Sensitivity", "Specificity", "F1", "Accuracy"]]
display(table.style.format({c: "{:.3f}" for c in ["Sensitivity", "Specificity", "F1", "Accuracy"]})
             .hide(axis="index"))

In [ ]:
FAMILY_COLOR = {"Radiomics": C_RAD, "Deep": C_DEEP, "Early fusion": C_FUSE, "Late fusion": C_FUSE}

r = results.iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 0.38 * len(r) + 1.2))
for i, row in r.iterrows():
    c = FAMILY_COLOR[row.Family]
    hollow = row.Family == "Late fusion"
    ax.plot([row.AUC_lo, row.AUC_hi], [i, i], color=c, lw=2, solid_capstyle="round")
    ax.scatter(row.AUC, i, s=60, color="white" if hollow else c, edgecolors=c, linewidths=2, zorder=3)
    ax.text(row.AUC_hi + 0.006, i, f"{row.AUC:.3f}", va="center", fontsize=8.5, color="#52514e")
ax.set_yticks(range(len(r)), r["Model"], fontsize=9)
ax.axvline(results.loc[results.Model == "Radiomics", "AUC"].item(), color=C_RAD, lw=1, ls="--", alpha=0.6)
ax.set_xlabel("Test AUC with patient-level bootstrap 95% CI")
ax.set_title("Held-out test performance", loc="left")
handles = [plt.Line2D([], [], marker="o", lw=2, color=c, markerfacecolor=fc, markersize=7, label=l)
           for l, c, fc in [("Radiomics", C_RAD, C_RAD), ("Deep", C_DEEP, C_DEEP),
                            ("Early fusion", C_FUSE, C_FUSE), ("Late fusion", C_FUSE, "white")]]
ax.legend(handles=handles, frameon=False, loc="lower right", fontsize=9)
ax.grid(axis="y", visible=False)
plt.tight_layout(); savefig("fig_test_auc_forest.png"); plt.show()

In [ ]:
best_arm = results.loc[0, "Model"]
roc_arms = list(dict.fromkeys([
    "Radiomics", f"{best_backbone} (frozen)", f"{best_backbone} (frozen) + Radiomics",
    f"{FT} (end-to-end)", f"{FT} + Radiomics", best_arm]))
styles = {"Radiomics": (C_RAD, "-"), "Deep": (C_DEEP, "-"), "Early fusion": (C_FUSE, "-"), "Late fusion": (C_FUSE, ":")}

fig, (ax, ax_cm) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.5, 1]})
for name in roc_arms:
    fpr, tpr, _ = roc_curve(y_te, test_probs[name])
    color, ls = styles[arm_family[name]]
    if "fine-tuned" in name and arm_family[name] != "Late fusion":
        ls = "--" if arm_family[name] == "Deep" else "-."
    ax.plot(fpr, tpr, color=color, ls=ls, lw=2, label=f"{name}  ({roc_auc_score(y_te, test_probs[name]):.3f})")
ax.plot([0, 1], [0, 1], color=C_MUTED, lw=1, ls=":")
ax.set(xlabel="False positive rate", ylabel="True positive rate", title="ROC — held-out test set")
ax.legend(loc="lower right", fontsize=8, frameon=False)

ConfusionMatrixDisplay(confusion_matrix(y_te, (test_probs[best_arm] >= 0.5).astype(int)),
                       display_labels=["Benign", "Malignant"]).plot(ax=ax_cm, cmap="Blues", colorbar=False)
ax_cm.set_title(f"Confusion matrix — {best_arm}", fontsize=10); ax_cm.grid(False)
plt.tight_layout(); savefig("fig_test_roc_confusion.png"); plt.show()

### 7.1 Is the fusion gain real? (paired bootstrap)

ΔAUC between two arms on the same bootstrap resamples. If the 95 % CI excludes 0 the difference is unlikely to be
noise from this particular test set.

In [ ]:
def paired_delta(a, b):
    d = boot_aucs(test_probs[a]) - boot_aucs(test_probs[b])
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    lo, hi = np.percentile(d, [2.5, 97.5])
    delta = roc_auc_score(y_te, test_probs[a]) - roc_auc_score(y_te, test_probs[b])
    return {"Comparison": f"{a}  vs  {b}", "ΔAUC": delta, "95% CI": f"{lo:+.3f} to {hi:+.3f}", "p (bootstrap)": p}

BB = f"{best_backbone} (frozen)"
comparisons = pd.DataFrame([
    paired_delta(f"{BB} + Radiomics", BB),
    paired_delta(f"{BB} + Radiomics", "Radiomics"),
    paired_delta(f"{FT} + Radiomics", f"{FT} (features)"),
    paired_delta(f"{FT} + Radiomics", "Radiomics"),
    paired_delta(f"Late fusion: {FT} + Radiomics", f"{FT} (end-to-end)"),
    paired_delta(f"{FT} (end-to-end)", BB),
])
comparisons.to_csv(f"{OUT_DIR}/paired_comparisons.csv", index=False)
display(comparisons.style.format({"ΔAUC": "{:+.3f}", "p (bootstrap)": "{:.3f}"}).hide(axis="index"))

### 7.2 Calcifications vs masses

In [ ]:
sub_arms = list(dict.fromkeys(["Radiomics", BB, f"{BB} + Radiomics", f"{FT} (end-to-end)", f"{FT} + Radiomics", best_arm]))
abn = df["abnormality_type"].to_numpy()[test_idx]
subgroup = pd.DataFrame({
    t: {arm: roc_auc_score(y_te[abn == t], test_probs[arm][abn == t]) for arm in sub_arms}
    for t in sorted(set(abn))
})
subgroup.columns = [f"{c} (n={int((abn == c).sum())})" for c in subgroup.columns]
display(subgroup.style.format("{:.3f}").background_gradient(cmap="Blues", axis=None))

### 7.3 What does the fused model rely on?

In [ ]:
fusion_arm = f"{FT} + Radiomics"
imp   = fitted[fusion_arm]["clf"].feature_importances_
n_rad = len(best_cols)
share = imp[:n_rad].sum() / imp.sum()

rad_only_imp = pd.Series(fitted["Radiomics"]["clf"].feature_importances_, index=best_cols).nlargest(12)[::-1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1, 2]})
ax1.bar(["Radiomics", "Deep"], [share, 1 - share], color=[C_RAD, C_DEEP], width=0.55)
for i, v in enumerate([share, 1 - share]):
    ax1.text(i, v + 0.01, f"{100 * v:.0f}%", ha="center", fontsize=10)
ax1.set_ylim(0, 1.1); ax1.set_ylabel("Share of total gain")
ax1.set_title(f"{fusion_arm}\n({n_rad} radiomics vs {imp.size - n_rad} deep features)", fontsize=10, loc="left")
ax1.grid(axis="x", visible=False)

ax2.barh([c.replace("original_", "") for c in rad_only_imp.index], rad_only_imp.values, color=C_RAD, height=0.6)
ax2.set_xlabel("XGBoost gain importance"); ax2.set_title("Top radiomics features (radiomics-only model)", loc="left", fontsize=10)
ax2.grid(axis="y", visible=False)
plt.tight_layout(); savefig("fig_feature_importance.png"); plt.show()

print(f"Radiomics features are {100 * n_rad / imp.size:.1f}% of the fused inputs but carry {100 * share:.1f}% of the gain.")

## 8. Key findings

In [ ]:
def auc_ci(name):
    r = results.set_index("Model").loc[name]
    return f"**{r.AUC:.3f}** (95% CI {r.AUC_lo:.3f}–{r.AUC_hi:.3f})"

def delta_line(i):
    c = comparisons.iloc[i]
    return f"ΔAUC {c['ΔAUC']:+.3f} (95% CI {c['95% CI']}, p = {c['p (bootstrap)']:.3f})"

def fmt_delta(d):
    return f"ΔAUC {d['ΔAUC']:+.3f} (95% CI {d['95% CI']}, p = {d['p (bootstrap)']:.3f})"

best_deep_frozen = results[results.Model.str.endswith("(frozen)")].iloc[0].Model
post_hoc = ""
if best_deep_frozen != BB:   # the test set favours a different backbone than CV picked → report it, flagged as post hoc
    post_hoc = (f"\n**Post hoc (chosen by looking at the test set, so optimistic):** {best_deep_frozen} + Radiomics reaches "
                f"AUC {auc_ci(f'{best_deep_frozen} + Radiomics')}; vs {best_deep_frozen} alone "
                f"{fmt_delta(paired_delta(f'{best_deep_frozen} + Radiomics', best_deep_frozen))}; vs radiomics alone "
                f"{fmt_delta(paired_delta(f'{best_deep_frozen} + Radiomics', 'Radiomics'))}.\n")

display(Markdown(f'''
**Pre-specified analysis** (backbone {best_backbone} selected by training-set CV before touching the test set):
- **Radiomics alone** ({len(best_cols)} features, selection: {best_fs}): AUC {auc_ci("Radiomics")}.
- **Early fusion with frozen {best_backbone}:** AUC {auc_ci(f"{BB} + Radiomics")} — vs deep alone {delta_line(0)}; vs radiomics alone {delta_line(1)}.
- **Fine-tuning {best_backbone}** (end-to-end): AUC {auc_ci(f"{FT} (end-to-end)")} — vs frozen {delta_line(5)}.
- **Fusion with the fine-tuned embedding:** AUC {auc_ci(f"{FT} + Radiomics")} — vs fine-tuned features alone {delta_line(2)}.
- In the fused model, radiomics make up {100 * n_rad / imp.size:.0f}% of the inputs but {100 * share:.0f}% of the XGBoost gain.

**All arms:** highest test AUC {best_arm} — {auc_ci(best_arm)}; best frozen deep features {best_deep_frozen} — {auc_ci(best_deep_frozen)}.
{post_hoc}'''))

with open(f"{OUT_DIR}/results_test.md", "w") as fh:
    fh.write("| Model | Family | AUC (95% CI) | Sens. | Spec. | F1 | Acc. |\n|---|---|---|---|---|---|---|\n")
    for _, r in results.iterrows():
        fh.write(f"| {r.Model} | {r.Family} | {r.AUC:.3f} ({r.AUC_lo:.3f}–{r.AUC_hi:.3f}) | "
                 f"{r.Sensitivity:.3f} | {r.Specificity:.3f} | {r.F1:.3f} | {r.Accuracy:.3f} |\n")
print(f"Saved: {sorted(os.listdir(OUT_DIR))}")

## 9. Discussion & limitations

**Interpretation.**
- Deep features from ImageNet backbones beat hand-crafted radiomics on this task. The best frozen
  backbone (DenseNet121) reached AUC ≈ 0.78 vs ≈ 0.74 for radiomics.
- **Fusion gives a small but consistent lift** of about +0.01 to +0.02 AUC over deep features alone. None of these
  gains reach significance on a test set of this size: every paired 95 % CI includes 0. In the fused model, the
  radiomics block carries a smaller share of the XGBoost gain than its share of the inputs. Most of the
  texture information the radiomics capture seems to be already present in the CNN embedding.
- **Partial fine-tuning did not significantly beat the frozen features** on ~1,900 training ROIs.
- **Model selection matters.** Training-set CV picked Swin-T, while the test set favoured DenseNet121. The gap between
  them lies within the bootstrap CIs, which is a reminder that the test-set ranking of individual backbones is noisy.

**Caveats.**

- **Leakage control.** Patient-grouped splits, selectors/scalers inside CV pipelines, class weights from the training set,
  and fine-tuning restricted to training patients mean the test numbers are an honest estimate. Earlier, row-level splits
  (and fine-tuning on images later used for evaluation) produced noticeably higher, over-optimistic AUCs.
- **Fine-tuned embeddings + XGBoost.** The XGBoost head is trained on embeddings of images the network was fine-tuned on,
  so those training embeddings are more separable than test embeddings; this can make the fused fine-tuned arm
  slightly over-confident. A cross-fitted (k-fold fine-tuning) scheme would remove that effect at ~5× the GPU cost.
- **Single held-out split.** The test set is ~20 % of patients; the bootstrap CIs show how much results move with
  the test sample. Repeated nested CV would give a tighter estimate.
- **Pre-cropped ROIs.** The task assumes the lesion has already been located; it is a characterisation model, not a detector.
- **Image source.** The Kaggle release is 8-bit JPEG converted from 16-bit DICOM; working from the original DICOMs may help both radiomics and CNNs.
- **Pooling calcifications and masses.** Subgroup AUCs (7.2) show how much performance differs between the two lesion types; separate models are a natural next step.
- **Threshold.** Sensitivity/specificity use a fixed 0.5 cut-off; a clinical deployment would choose an operating point (e.g. ≥ 90 % sensitivity) on validation data.

### Reproducing
Kaggle notebook with a GPU (T4), the image dataset `awsaf49/cbis-ddsm-breast-cancer-image-dataset` attached, and internet
enabled for the Hugging Face download. *Run All* takes roughly 25–40 minutes; cached features in `outputs/features/`
make re-runs much faster. All figures and result tables are written to `outputs/`.